# Cahier d'exercices SQL Vertica natif — à la banque Beobank

**Cahier PARTICIPANT — à compléter**
Formation Beobank · Orsys — SQL Vertica natif, niveau débutant

> ⚠️ **Notebook de référence, non exécuté dans cet environnement.** Il n'y a ici ni
> serveur Vertica accessible, ni Docker pour en simuler un. Le code ci-dessous utilise
> le pilote officiel **`vertica_python`** et se connecte à un **vrai cluster Vertica** —
> il a été relu attentivement pour la syntaxe (et repris du même dataset et des mêmes
> requêtes déjà validées sur SQLite dans `Cours_SQL.ipynb` / `Mises_En_Situation_Bancaires_SQL.ipynb`),
> mais n'a pas pu être testé par une exécution réelle. **À tester sur un vrai cluster
> Vertica Beobank avant utilisation en formation.**
>
> Contrairement aux notebooks SQLite du dossier `cours_sql/`, on ne charge PAS les CSV
> en Python : dans un vrai environnement Vertica, les tables `CTR`, `TIE`, `TIE_ADR`,
> `TIE_X_CTR`, `TXN_X_CTR` existent déjà dans la base (chargées en amont par un ETL ou
> une commande `COPY`). On se contente ici de s'y CONNECTER et de les interroger.

Ce notebook réutilise les mêmes 5 tables et les mêmes questions bancaires que la version
SQLite, mais avec de vraies fonctions **natives Vertica** (`DATEDIFF`, `TO_CHAR`, `ILIKE`...)
au lieu des équivalents SQLite (`julianday`, `strftime`, `LIKE`) utilisés uniquement parce
qu'aucun serveur Vertica n'était disponible pour l'exécution locale.

## Petit glossaire des colonnes du dataset

| Colonne | Table | Ce qu'elle représente |
|---|---|---|
| `NUM_TIE` / `IDT_PI` | TIE, TIE_ADR, TIE_X_CTR | Numéro qui identifie un client de façon unique |
| `COD_TYP_TIE` | TIE | Type de client : 1 = particulier, 2 = entreprise |
| `DAT_NAI` | TIE | Date de naissance du client |
| `COD_LNG_CTR` | TIE | Langue de contact du client (FR ou NL) |
| `IDT_AC` | CTR, TIE_X_CTR, TXN_X_CTR | Numéro qui identifie un compte bancaire de façon unique |
| `DAT_OUV_CTR` / `DAT_CLO_CTR` | CTR | Date d'ouverture / de clôture du compte |
| `COD_ECV_CTR` | CTR | Code d'état du compte : 4 = actif, 6 = clôturé |
| `COD_DEV` | CTR | Devise du compte (EUR, USD...) |
| `SLD_CTR` | CTR | Solde du compte (l'argent qu'il y a dessus) |
| `ADR_EMA` | TIE_ADR | Adresse email du client |
| `NUM_TEL_MOB_INL` / `NUM_TEL_DOM_INL` | TIE_ADR | Téléphone mobile / fixe du client |
| `LIB_OPE_INL_1` | TXN_X_CTR | Libellé (texte qui décrit) une opération bancaire |
| `DAT_MVT` / `MNT_MVT` | TXN_X_CTR | Date / montant du mouvement (l'extrait CSV pédagogique n'a pas de vrai montant : `MNT_MVT` représente la colonne de montant qui existe réellement dans l'entrepôt Vertica de production — à renommer selon le nom réel de votre table) |

In [ ]:
import os                            # pour lire les identifiants depuis des variables d'environnement
import pandas as pd                  # pour récupérer les résultats SQL sous forme de tableau

# vertica_python : le pilote Python OFFICIEL pour Vertica -- respecte la même interface
# standard (DB-API 2.0) que sqlite3, vu dans les autres notebooks de ce cours
# Installation dans un projet réel :  pip install vertica-python
import vertica_python

# Paramètres de connexion -- À ADAPTER à votre cluster Vertica Beobank réel.
# RÈGLE DE SÉCURITÉ : ne jamais écrire un mot de passe en clair dans le code.
# En pratique : variables d'environnement (comme ici), ou un fichier de config non versionné.
parametres_connexion = {
    "host":       "vertica.beobank.local",              # adresse du serveur (ou de l'équilibreur de charge)
    "port":       5433,                                  # port par défaut de Vertica
    "user":       os.environ.get("VERTICA_USER", "moi"),
    "password":   os.environ.get("VERTICA_PWD", ""),     # lu depuis une variable d'environnement, jamais en dur
    "database":   "beobank_dwh",                          # nom de la base analytique
    "autocommit": False,                                  # écriture validée par commit() explicite, comme sqlite3
}

try:
    conn = vertica_python.connect(**parametres_connexion)   # connect() : ouvre la connexion réseau vers Vertica
    print("Connecté à Vertica :", parametres_connexion["host"])
except Exception as erreur:
    # Dans CET environnement de formation, aucun cluster Vertica n'est accessible -- c'est
    # attendu. Le code ci-dessus reste la RÉFÉRENCE exacte à utiliser une fois en poste,
    # contre le vrai serveur Vertica Beobank (identifiants réels, réseau de l'entreprise).
    conn = None
    print("Connexion Vertica impossible ici (normal dans cet environnement) :", erreur)
    print("Ce notebook sert de modèle de référence pour un vrai cluster Vertica.")

> ⚠️ **Non exécuté ici** — pas de cluster Vertica accessible dans cet environnement de
> génération (ni serveur, ni Docker). Requêtes relues attentivement pour la syntaxe, et
> déjà validées par exécution réelle dans leur équivalent SQLite
> (`cours_sql/Cahier_Corrige_Formateur_SQL.ipynb`). **À tester sur un vrai cluster Vertica
> avant utilisation en formation.**

**⚠️ Exécutez les cellules dans l'ordre.** Le code de connexion ci-dessus doit être exécuté une seule fois, puis chaque exercice peut être complété.

**Comment compléter :** chaque étape explique en détail ce qu'il faut faire et pourquoi (comme le ferait le formateur), avec la requête déjà presque entièrement écrite et commentée : il ne vous manque que **trois mots-clés à trouver par cellule**, marqués par `______`. Le commentaire SQL (`--`) juste au-dessus vous dit ce qu'il faut y mettre.

---
## Exercice 1 — Premier jour : explorer la table des clients

**Mise en situation :** C'est votre premier jour comme data analyst SQL chez Beobank. Votre chef d'équipe vous dit : « Regarde ce qu'il y a dans la table des clients. »

**Étape 1.1.** Affichez les 5 premières lignes de la table `TIE` (les clients) avec **`SELECT * ... LIMIT 5`**. *(Trois éléments à compléter dans la cellule ci-dessous.)*

In [ ]:
requete = """
-- SELECT * : on demande TOUTES les colonnes
-- FROM TIE : la table des clients ("TIE" = "tiers", le mot bancaire pour "client")
-- LIMIT 5 : on ne veut que les 5 premières lignes, pour un premier aperçu
______ * ______ TIE ______ 5
"""
print(pd.read_sql(requete, conn))

**Étape 1.2.** Comptez le nombre total de clients avec **`COUNT(*)`**. *(Trois éléments à compléter dans la cellule ci-dessous.)*

In [ ]:
requete = """
-- COUNT(*) compte le nombre de lignes de la table
-- AS nb_clients : on donne un nom clair à la colonne du résultat (un alias)
______ ______(*) AS nb_clients ______ TIE
"""
print(pd.read_sql(requete, conn))

**Étape 1.3.** Listez les langues de contact différentes présentes dans `TIE` avec **`DISTINCT`**. *(Trois éléments à compléter dans la cellule ci-dessous.)*

In [ ]:
requete = """
-- COD_LNG_CTR = la langue de contact du client (FR ou NL)
-- DISTINCT enlève les doublons : chaque valeur n'apparaît qu'une fois dans le résultat
______ ______ COD_LNG_CTR ______ TIE
"""
print(pd.read_sql(requete, conn))

---
## Exercice 2 — Préparer un mailing marketing

**Mise en situation :** Le service marketing veut savoir : « Est-ce qu'on a surtout des particuliers ou des entreprises ? »

**Étape 2.1.** Traduisez le code `COD_TYP_TIE` en texte lisible avec **`CASE WHEN`** (1 = Particulier, 2 = Entreprise). *(Trois éléments à compléter dans la cellule ci-dessous.)*

In [ ]:
requete = """
-- COD_TYP_TIE = code type de client (1 = personne physique, 2 = personne morale)
-- CASE WHEN condition THEN valeur ELSE autre_valeur END : le "SI... ALORS... SINON" du SQL
______ NUM_TIE,
       ______ WHEN COD_TYP_TIE = 1 THEN 'Particulier' ELSE 'Entreprise' END AS type_client
______ TIE
LIMIT 5
"""
print(pd.read_sql(requete, conn))

**Étape 2.2.** Comptez le nombre de clients par type avec **`GROUP BY`**. *(Trois éléments à compléter dans la cellule ci-dessous.)*

In [ ]:
requete = """
-- GROUP BY type_client : regroupe toutes les lignes qui ont la même valeur de type_client
-- COUNT(*) est alors calculé SÉPARÉMENT pour chaque groupe
SELECT
    ______ WHEN COD_TYP_TIE = 1 THEN 'Particulier' ELSE 'Entreprise' END AS type_client,
    ______(*) AS nb_clients
FROM TIE
______ type_client
"""
print(pd.read_sql(requete, conn))

**Étape 2.3.** Faites la même chose sur `COD_LNG_CTR` (la langue de contact). *(Trois éléments à compléter dans la cellule ci-dessous.)*

In [ ]:
requete = """
-- même principe : un groupe par langue de contact (FR / NL), et un comptage par groupe
______ COD_LNG_CTR, ______(*) AS nb_clients
FROM TIE
______ COD_LNG_CTR
"""
print(pd.read_sql(requete, conn))

---
## Exercice 3 — Traduire le statut des comptes

**Mise en situation :** Un collègue ouvre la table des comptes (`CTR`) et ne comprend pas les codes : « `COD_ECV_CTR` égal à 4 ou 6, ça veut dire quoi ? »

**Étape 3.1.** Affichez `IDT_AC` et `COD_ECV_CTR` traduit en texte avec **`CASE WHEN`** (4 = Actif, 6 = Clôturé). *(Trois éléments à compléter dans la cellule ci-dessous.)*

In [ ]:
requete = """
-- IDT_AC = identifiant du compte bancaire
-- COD_ECV_CTR = code d'état du compte : 4 = actif, 6 = clôturé
-- CASE WHEN peut avoir plusieurs WHEN à la suite, comme un SI / SINON SI / SINON
______ IDT_AC,
       ______ WHEN COD_ECV_CTR = 4 THEN 'Actif' WHEN COD_ECV_CTR = 6 THEN 'Clôturé' END AS statut
______ CTR
LIMIT 5
"""
print(pd.read_sql(requete, conn))

**Étape 3.2.** Comptez le nombre de comptes actifs vs clôturés, en réutilisant ce `CASE WHEN` dans un **`GROUP BY`**. *(Trois éléments à compléter dans la cellule ci-dessous.)*

In [ ]:
requete = """
-- on regroupe par le statut qu'on vient de calculer, et on compte les comptes de chaque groupe
SELECT
    ______ WHEN COD_ECV_CTR = 4 THEN 'Actif' WHEN COD_ECV_CTR = 6 THEN 'Clôturé' END AS statut,
    ______(*) AS nb_comptes
FROM CTR
______ statut
"""
print(pd.read_sql(requete, conn))

**Étape 3.3.** Trouvez les comptes dont le solde `SLD_CTR` est manquant avec **`IS NULL`**. *(Trois éléments à compléter dans la cellule ci-dessous.)*

In [ ]:
requete = """
-- SLD_CTR = solde du compte. IS NULL teste si une valeur est MANQUANTE
-- (on ne peut jamais écrire "= NULL" en SQL, NULL se teste avec IS NULL / IS NOT NULL)
SELECT IDT_AC FROM CTR ______ SLD_CTR ______ ______ 5
"""
print(pd.read_sql(requete, conn))

---
## Exercice 4 — Repérer les comptes en découvert

**Mise en situation :** Le service risque prépare un comité de crédit demain matin. Il a besoin de la liste des comptes dont le solde est négatif.

**Étape 4.1.** Sélectionnez les comptes en découvert (`SLD_CTR < 0`) avec **`WHERE`**. *(Trois éléments à compléter dans la cellule ci-dessous.)*

In [ ]:
requete = """
-- WHERE filtre les LIGNES : on ne garde que celles où SLD_CTR est strictement négatif
-- COD_DEV = devise du compte (EUR, USD...)
______ IDT_AC, SLD_CTR, COD_DEV FROM CTR ______ SLD_CTR ______ 0
"""
print(pd.read_sql(requete, conn))

**Étape 4.2.** Triez ces comptes du découvert le plus important au moins important avec **`ORDER BY`**. *(Trois éléments à compléter dans la cellule ci-dessous.)*

In [ ]:
requete = """
-- ORDER BY SLD_CTR ASC : tri croissant (ASC = ascending), donc les valeurs les plus
-- négatives (les pires découverts) apparaissent en premier
SELECT IDT_AC, SLD_CTR FROM CTR ______ SLD_CTR < 0 ______ SLD_CTR ______
"""
print(pd.read_sql(requete, conn))

**Étape 4.3.** Affichez les 5 plus gros soldes positifs avec `ORDER BY ... DESC` et **`LIMIT`**. *(Trois éléments à compléter dans la cellule ci-dessous.)*

In [ ]:
requete = """
-- DESC (descending) : tri décroissant, les plus gros soldes en premier
-- LIMIT 5 : on garde seulement les 5 premières lignes du résultat trié
SELECT IDT_AC, SLD_CTR FROM CTR ______ SLD_CTR ______ ______ 5
"""
print(pd.read_sql(requete, conn))

**Étape 4.4.** Calculez le montant total en découvert avec **`SUM()`**. *(Trois éléments à compléter dans la cellule ci-dessous.)*

In [ ]:
requete = """
-- SUM(SLD_CTR) additionne tous les soldes des lignes gardées par le WHERE
-- comme ce sont des soldes négatifs, le résultat est un grand nombre négatif
SELECT ______(SLD_CTR) AS total_decouvert ______ CTR ______ SLD_CTR < 0
"""
print(pd.read_sql(requete, conn))

---
## Exercice 5 — Protéger les clients âgés

**Mise en situation :** Le service conformité doit surveiller particulièrement les clients de 75 ans et plus.

**Étape 5.1.** Calculez l'âge de chaque client en années avec la fonction native Vertica **`DATEDIFF('year', ..., CURRENT_DATE)`**. *(Trois éléments à compléter dans la cellule ci-dessous.)*

In [ ]:
requete = """
-- DAT_NAI = date de naissance du client
-- DATEDIFF('year', date_debut, date_fin) : fonction Vertica native qui calcule
-- directement le nombre d'années entre deux dates (pas besoin de calcul manuel)
-- CURRENT_DATE : la date du jour, fournie par le serveur Vertica
______ NUM_TIE, DAT_NAI,
       ______('year', DAT_NAI, CURRENT_DATE) AS age
______ TIE
LIMIT 5
"""
print(pd.read_sql(requete, conn))

**Étape 5.2.** Gardez uniquement les clients de 75 ans et plus, en réutilisant ce calcul dans un **`WHERE`**. *(Trois éléments à compléter dans la cellule ci-dessous.)*

In [ ]:
requete = """
-- on répète le même calcul d'âge dans le WHERE, avec la condition >= 75
SELECT NUM_TIE, DATEDIFF('year', DAT_NAI, CURRENT_DATE) AS age
______ TIE
______ DATEDIFF('year', DAT_NAI, CURRENT_DATE) ______ 75
"""
print(pd.read_sql(requete, conn))

**Étape 5.3.** Comptez ces clients avec **`COUNT(*)`**. *(Trois éléments à compléter dans la cellule ci-dessous.)*

In [ ]:
requete = """
-- même filtre WHERE que juste au-dessus, mais on ne demande que le COMPTAGE cette fois
SELECT ______(*) AS nb_clients_ages
______ TIE
______ DATEDIFF('year', DAT_NAI, CURRENT_DATE) >= 75
"""
print(pd.read_sql(requete, conn))

---
## Exercice 6 — Vérifier les coordonnées des clients

**Mise en situation :** Avant une campagne email, le marketing veut savoir : « Est-ce qu'on a bien les coordonnées de tout le monde ? »

**Étape 6.1.** Comptez les clients qui ONT un email, avec **`IS NOT NULL`**. *(Trois éléments à compléter dans la cellule ci-dessous.)*

In [ ]:
requete = """
-- ADR_EMA = adresse email du client, dans la table TIE_ADR
-- IS NOT NULL teste si la valeur EST renseignée (pas manquante)
SELECT ______(*) AS nb_avec_email ______ TIE_ADR WHERE ADR_EMA ______
"""
print(pd.read_sql(requete, conn))

**Étape 6.2.** Trouvez les clients sans email NI téléphone (mobile ou fixe), en combinant plusieurs `IS NULL` avec **`AND`**. *(Trois éléments à compléter dans la cellule ci-dessous.)*

In [ ]:
requete = """
-- NUM_TEL_MOB_INL = téléphone mobile, NUM_TEL_DOM_INL = téléphone fixe
-- AND veut dire "et" : les TROIS conditions doivent être vraies en même temps
SELECT NUM_TIE, NOM_TIE
______ TIE_ADR
______ ADR_EMA IS NULL ______ NUM_TEL_MOB_INL IS NULL AND NUM_TEL_DOM_INL IS NULL
"""
print(pd.read_sql(requete, conn))

**Étape 6.3.** Recherchez les clients dont l'email contient `"gmail"` avec **`ILIKE`** (recherche de texte insensible à la casse). *(Trois éléments à compléter dans la cellule ci-dessous.)*

In [ ]:
requete = """
-- ILIKE recherche un motif dans du texte, SANS tenir compte des majuscules/minuscules
-- (contrairement à LIKE, qui est sensible à la casse sur Vertica)
-- le symbole % veut dire "n'importe quel texte avant/après"
SELECT NUM_TIE, ADR_EMA ______ TIE_ADR ______ ADR_EMA ______ '%gmail%'
"""
print(pd.read_sql(requete, conn))

---
## Exercice 7 — Enquêter sur un client qui a beaucoup de comptes

**Mise en situation :** Le directeur régional remarque qu'un client semble avoir énormément de comptes. Il vous demande de vérifier.

**Étape 7.1.** Comptez le nombre de comptes par client avec **`GROUP BY`** sur `TIE_X_CTR`. *(Trois éléments à compléter dans la cellule ci-dessous.)*

In [ ]:
requete = """
-- TIE_X_CTR = la table qui relie chaque client (NUM_TIE) à ses comptes (IDT_AC)
-- un groupe par client, et on compte les lignes (donc les comptes) de chaque groupe
SELECT NUM_TIE, ______(*) AS nb_comptes
FROM TIE_X_CTR
______ NUM_TIE
______ nb_comptes DESC
"""
print(pd.read_sql(requete, conn))

**Étape 7.2.** Gardez uniquement les clients qui ont plus de 5 comptes, avec **`HAVING`** (le `WHERE` des groupes). *(Trois éléments à compléter dans la cellule ci-dessous.)*

In [ ]:
requete = """
-- WHERE filtre les LIGNES avant regroupement ; HAVING filtre les GROUPES après le COUNT
-- on ne peut pas écrire WHERE COUNT(*) > 5 : il FAUT HAVING pour filtrer un résultat agrégé
SELECT NUM_TIE, COUNT(*) AS nb_comptes
FROM TIE_X_CTR
______ NUM_TIE
______ COUNT(*) > 5
______ nb_comptes DESC
"""
print(pd.read_sql(requete, conn))

**Étape 7.3.** Trouvez les clients de `TIE` absents de `TIE_X_CTR` (aucun compte) avec **`NOT IN`** et une sous-requête. *(Trois éléments à compléter dans la cellule ci-dessous.)*

In [ ]:
requete = """
-- (SELECT NUM_TIE FROM TIE_X_CTR) est une SOUS-REQUÊTE : elle renvoie la liste de
-- tous les NUM_TIE qui ONT un compte
-- NOT IN (...) garde les clients dont le NUM_TIE n'est PAS dans cette liste
SELECT NUM_TIE ______ TIE
______ NUM_TIE ______ (SELECT NUM_TIE FROM TIE_X_CTR)
"""
print(pd.read_sql(requete, conn))

---
## Exercice 8 — Répondre à un client qui conteste des frais

**Mise en situation :** Un client appelle, énervé : « Il y a plein de prélèvements que je ne comprends pas sur mon compte `65500477817` ! »

**Étape 8.1.** Affichez toutes les opérations de ce compte, triées par date, avec **`WHERE`** et `ORDER BY`. *(Trois éléments à compléter dans la cellule ci-dessous.)*

In [ ]:
requete = """
-- LIB_OPE_INL_1 = libellé qui décrit l'opération, MNT_MVT = montant du mouvement
-- WHERE IDT_AC = ... : uniquement les opérations de ce compte précis
-- ORDER BY DAT_MVT : de la plus ancienne à la plus récente
SELECT DAT_MVT, LIB_OPE_INL_1, MNT_MVT
______ TXN_X_CTR
______ IDT_AC = 65500477817
______ DAT_MVT
"""
print(pd.read_sql(requete, conn))

**Étape 8.2.** Filtrez seulement les frais avec **`ILIKE`** sur le libellé (insensible à la casse : "Frais" ou "frais" seront trouvés). *(Trois éléments à compléter dans la cellule ci-dessous.)*

In [ ]:
requete = """
-- ILIKE '%frais%' : le libellé contient "frais" n'importe où dans le texte,
-- quelle que soit la casse (trouve aussi bien "Frais" que "frais")
SELECT DAT_MVT, LIB_OPE_INL_1, MNT_MVT
FROM TXN_X_CTR
______ IDT_AC = 65500477817 ______ LIB_OPE_INL_1 ______ '%frais%'
ORDER BY DAT_MVT
"""
print(pd.read_sql(requete, conn))

**Étape 8.3.** Calculez le total des frais de ce compte avec **`SUM()`**. *(Trois éléments à compléter dans la cellule ci-dessous.)*

In [ ]:
requete = """
-- SUM(MNT_MVT) additionne les montants de toutes les opérations de frais trouvées
SELECT ______(MNT_MVT) AS total_frais
______ TXN_X_CTR
WHERE IDT_AC = 65500477817 AND LIB_OPE_INL_1 ______ '%frais%'
"""
print(pd.read_sql(requete, conn))

---
## Exercice 9 — Prévoir l'activité du centre d'appel

**Mise en situation :** Le service RH doit décider combien de conseillers embaucher. Il demande quel mois a le plus d'opérations.

**Étape 9.1.** Regroupez les opérations par mois avec la fonction native Vertica **`TO_CHAR(DAT_MVT, 'YYYY-MM')`** (formate une date en texte "année-mois"). *(Trois éléments à compléter dans la cellule ci-dessous.)*

In [ ]:
requete = """
-- TO_CHAR(DAT_MVT, 'YYYY-MM') transforme chaque date en texte "année-mois" (ex: 2026-04)
-- GROUP BY mois : un groupe par mois, COUNT(*) compte les opérations de chaque mois
SELECT ______(DAT_MVT, 'YYYY-MM') AS mois, ______(*) AS nb_operations
FROM TXN_X_CTR
______ mois
ORDER BY mois
"""
print(pd.read_sql(requete, conn))

**Étape 9.2.** Affichez le mois le plus chargé en premier avec **`ORDER BY ... DESC`**. *(Trois éléments à compléter dans la cellule ci-dessous.)*

In [ ]:
requete = """
-- ORDER BY nb_operations DESC : tri décroissant sur le comptage
-- LIMIT 1 : on ne garde que le tout premier -> le mois le plus chargé
SELECT TO_CHAR(DAT_MVT, 'YYYY-MM') AS mois, COUNT(*) AS nb_operations
FROM TXN_X_CTR
______ mois
______ nb_operations ______
LIMIT 1
"""
print(pd.read_sql(requete, conn))

---
## Exercice 10 — Construire une fiche client à 360°

**Mise en situation :** Un conseiller a un rendez-vous avec un client important. Il veut voir en une fois tous ses comptes et leur solde.

**Étape 10.1.** Fusionnez `TIE`, `TIE_X_CTR` et `CTR` avec **`JOIN`**, sur les bonnes colonnes communes. *(Trois éléments à compléter dans la cellule ci-dessous.)*

In [ ]:
requete = """
-- JOIN table ON condition : rapproche deux tables qui ont une colonne en commun
-- t, x, c sont des ALIAS (raccourcis) pour TIE, TIE_X_CTR et CTR
-- ici on enchaîne deux JOIN : TIE -> TIE_X_CTR (par NUM_TIE) -> CTR (par IDT_AC)
SELECT t.NUM_TIE, c.IDT_AC, c.SLD_CTR, c.COD_DEV
______ TIE t
______ TIE_X_CTR x ______ t.NUM_TIE = x.NUM_TIE
JOIN CTR c ON x.IDT_AC = c.IDT_AC
LIMIT 5
"""
print(pd.read_sql(requete, conn))

**Étape 10.2.** Calculez le solde total par client avec **`GROUP BY`** et `SUM()`, pour le client `2900000004654`. *(Trois éléments à compléter dans la cellule ci-dessous.)*

In [ ]:
requete = """
-- on ajoute une clause GROUP BY pour agréger tous les comptes d'un même client
-- WHERE filtre AVANT le regroupement : seulement les lignes de ce client précis
SELECT t.NUM_TIE, ______(c.SLD_CTR) AS solde_total, COUNT(c.IDT_AC) AS nb_comptes
FROM TIE t
JOIN TIE_X_CTR x ON t.NUM_TIE = x.NUM_TIE
JOIN CTR c ON x.IDT_AC = c.IDT_AC
______ t.NUM_TIE = 2900000004654
______ t.NUM_TIE
"""
print(pd.read_sql(requete, conn))

**Étape 10.3.** Créez une **`VIEW`** (vue) réutilisable pour cette fiche 360°, sans le filtre `WHERE`. *(Trois éléments à compléter dans la cellule ci-dessous.)*

In [ ]:
curseur = conn.cursor()
for instruction in """
-- CREATE VIEW nom AS requête : enregistre la requête sous un nom, comme une table virtuelle
-- ensuite, "SELECT * FROM vue_360" relance la requête complète sans la réécrire
-- DROP VIEW IF EXISTS : supprime la vue si elle existe déjà (pour pouvoir relancer la cellule)
______ IF EXISTS vue_360;
______ vue_360 AS
SELECT t.NUM_TIE, SUM(c.SLD_CTR) AS solde_total, COUNT(c.IDT_AC) AS nb_comptes
FROM TIE t
JOIN TIE_X_CTR x ON t.NUM_TIE = x.NUM_TIE
JOIN CTR c ON x.IDT_AC = c.IDT_AC
______ t.NUM_TIE;
""".split(";"):
    if instruction.strip():             # ignore les morceaux vides (après le dernier ";")
        curseur.execute(instruction)    # une instruction à la fois : pas d'executescript() en DB-API 2.0
conn.commit()                            # valide définitivement la création de la vue
curseur.close()
print("Vue créée.")

**Étape 10.4.** Interrogez la vue pour retrouver les 5 plus gros patrimoines avec un simple **`SELECT`**. *(Trois éléments à compléter dans la cellule ci-dessous.)*

In [ ]:
requete = """
-- la vue vue_360 se lit comme une table normale, alors qu'elle rejoue en fait
-- la requête à 3 tables définie juste au-dessus
SELECT * ______ vue_360 ______ solde_total ______ LIMIT 5
"""
print(pd.read_sql(requete, conn))

---
## Fin du cahier

Comparez vos réponses avec le cahier corrigé du formateur (`Cahier_Corrige_Formateur_Vertica.ipynb`).